# Forecasting GDP Growth
This project tasks you with forecasting quarterly nominal GDP growth rate in the United States. The dataset contains estimated nominal gross domestic product (GDP) annual growth rate in each quarter from 1990 to 2020, released by the Bureau of Economic Analysis (BEA) of the United States Department of Commerce. The estimate is the “third” or "final" estimate of GDP issued in the third month after the relevant quarter which is based on more complete source data than were available for the "second" estimate issued in the second month and the "advance" estimate issued in the first month after the relevant quarter. You may refer to the [BEA website](https://www.bea.gov/resources/learning-center/what-to-know-gdp) for more information on the estimate of GDP in the United States. You will train models to forecast the BEA's final GDP estimate for a quarter. Specifically you are required to answer this question: Is accounting information useful for GDP forecast? You will be allowed to use any outside data to build your model.

The evaluation pages describes how submissions will be scored and how students should format their submissions. --- The evaluation metric for this competition is [RMSE](https://en.wikipedia.org/wiki/Root-mean-square_deviation). RMSE is a measure of the accuracy of your predictions.

**Files**
- GDP_Forecast_train.csv - the training set
- GDP_Forecast_test.csv - the test set
- GDP_Forecast_sampleSubmission.csv - a sample submission file in the correct format

**Columns**
- YQ - The year and quarter variable. 1990Q1 refers to the first calendar quarter of 1990, ie, Jan - March 1990
- NGDP - nominal GDP growth rate in the quarter of the year. It is obtained from the BEA final estimate which is available at the BEA website.

**Features** (no features data provided)

Following Datar et al. (2020), you may consider the following data as your predictors/features:
- Accounting data - through COMPUSTAT on WRDS
- Stock price/return data - through CRSP on WRDS
- Analysts' forecast of GDP - through Survey of Professional Forecasters
- Any other data

In [32]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from sklearn.metrics import mean_squared_error

train = pd.read_csv("data/GDP_Forecast_train.csv")
test = pd.read_csv("data/GDP_Forecast_test.csv")

def create_submission(test_df, y_pred, filename):
    df = pd.DataFrame({"YQ": test_df["YQ"], "NGDP": y_pred})
    df.to_csv(f"attempts/{filename}.csv", index = False)

## splitting year-quarter to check for traits in yearly patterns vs quarterly pattern
# ensure year is an integer
train["year"] = train["YQ"].str[:4].astype(int)
test["year"] = test["YQ"].str[:4].astype(int)
# keep qtr as string / category
train["qtr"] = train["YQ"].str[4:]
test["qtr"] = test["YQ"].str[4:]
train.head()

,YQ,NGDP,year,qtr
0,1990Q1,9.0,1990,Q1
1,1990Q2,6.1,1990,Q2
2,1990Q3,3.7,1990,Q3
3,1990Q4,-0.7,1990,Q4
4,1991Q1,2.0,1991,Q1


### Base Model

In [34]:
# running base linear model
base_model = smf.ols("NGDP ~ year + qtr", train).fit()
# printing summary of base_model
print(base_model.summary())

# get predicted values based off model
y_pred = base_model.predict(test)
y_true = test["NGDP"]
create_submission(test, y_pred, "0_base_forecast")
# evaluate using RMSE:
print(f"\nRMSE: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     3.324
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0134
Time:                        01:42:22   Log-Likelihood:                -244.91
No. Observations:                 104   AIC:                             499.8
Df Residuals:                      99   BIC:                             513.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    219.8404     68.424      3.213      0.0

### How Accounting Data affects GDP (COMPUSTAT on WRDS)
- [Compustat North America](https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/north-america-daily/) is a database of U.S. and Canadian fundamental and market information on active and inactive publicly held companies. It provides more than 300 annual and 100 quarterly Income Statement, Balance Sheet, Statement of Cash Flows, and supplemental data items. For most companies, annual history is available back to 1950 and quarterly history back to 1962 with monthly market history back to 1962.
- To consider [Execucomp](https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/execucomp/): Executive compensation data collected directly from each company’s annual proxy (DEF14A SEC form)



In [35]:
# Using Compustat North America: Fundamentals Quarterly
# https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/north-america-daily/fundamentals-quarterly/
# NOTE: currency in USD, data in STD format

compustat = pd.read_csv("data/compustat_quarterly_financials.csv")
## dropping columns with no unique data
compustat.drop(columns = ["curcdq", "datafmt", "indfmt", "consol"], inplace = True)
## dropping columns without too many missing data
# # print(compustat[compustat["ltq"].notna()].shape) # 26666 out of 85349
# compustat.drop(columns = ["ltq"], inplace = True)
## droping costat - only 2 options
compustat.drop(columns = ["costat"], inplace = True)
compustat.head()

,gvkey,datadate,conm,datacqtr,atq,ltq,niq
0,1178,1990-03-31,AFFILIATED BANKSHARES COLO,1990Q1,2597.647,NaN,2.445
1,1178,1990-06-30,AFFILIATED BANKSHARES COLO,1990Q2,2645.271,NaN,3.234
2,1178,1990-09-30,AFFILIATED BANKSHARES COLO,1990Q3,2682.906,NaN,3.253
3,1178,1990-12-31,AFFILIATED BANKSHARES COLO,1990Q4,2738.097,NaN,0.897
4,1178,1991-03-31,AFFILIATED BANKSHARES COLO,1991Q1,2655.894,NaN,3.536


- (gvkey) Global Company Key
- (conm) Company Name
- (datacqtr) Calendar Data Year and Quarter
- (atq) Assets - Total
- (ltq) Liabilities - Total
- (niq) Net Income (Loss)

In [36]:
# merge compustat data to train and test
train_comp_macro = train.merge(compustat, how = "left", left_on = "YQ", right_on = "datacqtr")
test_comp_macro = test.merge(compustat, how = "left", left_on = "YQ", right_on = "datacqtr")
train_comp_macro.head()

,YQ,NGDP,year,qtr,gvkey,datadate,conm,datacqtr,atq,ltq,niq
0,1990Q1,9.0,1990,Q1,1178,1990-03-31,AFFILIATED BANKSHARES COLO,1990Q1,2597.647,NaN,2.445
1,1990Q1,9.0,1990,Q1,1194,1990-03-31,AHMANSON (H F) & CO,1990Q1,45984.500,NaN,65.978
2,1990Q1,9.0,1990,Q1,1416,1990-03-31,AMERICAN CAPITAL CORP,1990Q1,5257.766,NaN,NaN
3,1990Q1,9.0,1990,Q1,1553,1990-03-31,AMERICAN SVGS/FL FSB,1990Q1,NaN,NaN,0.698
4,1990Q1,9.0,1990,Q1,3431,1990-03-31,CONSTELLATION BANCORP,1990Q1,NaN,NaN,NaN


In [37]:
# create base compustat model
base_comp_model = smf.ols("NGDP ~ year + qtr + conm + atq + ltq + niq", train_comp_macro).fit()
print(base_comp_model.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.194
Model:                            OLS   Adj. R-squared:                  0.153
Method:                 Least Squares   F-statistic:                     4.677
Date:                Wed, 04 Mar 2026   Prob (F-statistic):               0.00
Time:                        01:42:49   Log-Likelihood:                -65856.
No. Observations:               26636   AIC:                         1.343e+05
Df Residuals:                   25329   BIC:                         1.450e+05
Df Model:                        1306                                         
Covariance Type:            nonrobust                                         
                                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------


Insights: too many companies, run time too long, with all the companies, Assets is highly insignificant (p-value: 0.983)

In [ ]:
## create aggregates for assets, liabilities and net income
compustat_macro = compustat.groupby("datacqtr").agg({"atq": "sum", "ltq": "sum", "niq": "sum"}).reset_index()
compustat_macro.sort_values(by = "datacqtr")

# calculate lag
compustat_macro["atq_lag"] = compustat_macro["atq"].shift(1)
compustat_macro["ltq_lag"] = compustat_macro["ltq"].shift(1)
compustat_macro["niq_lag"] = compustat_macro["niq"].shift(1)
# calculate growth
compustat_macro["atq_growth"] = compustat_macro["atq"] / compustat_macro["atq_lag"] - 1
compustat_macro["ltq_growth"] = compustat_macro["ltq"] / compustat_macro["ltq_lag"] - 1
compustat_macro["niq_growth"] = compustat_macro["niq"] / compustat_macro["niq_lag"] - 1
# converting infinite values to NaN
compustat_macro = compustat_macro.replace([np.inf, -np.inf], np.nan)
compustat_macro.head()

,datacqtr,atq,ltq,niq,atq_lag,ltq_lag,niq_lag,atq_growth,ltq_growth,niq_growth
0,1989Q4,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN
1,1990Q1,3187273.043,224746.034,4684.085,0.000,0.000,0.000,NaN,NaN,NaN
2,1990Q2,3412321.689,236377.703,6396.105,3187273.043,224746.034,4684.085,0.070609,0.051755,0.365497
3,1990Q3,3473969.743,239062.003,3093.203,3412321.689,236377.703,6396.105,0.018066,0.011356,-0.516393
4,1990Q4,3426461.019,245699.448,-49.699,3473969.743,239062.003,3093.203,-0.013676,0.027765,-1.016067


In [39]:
# merge compustat data to train and test
train_comp_macro = train.merge(compustat_macro, how = "left", left_on = "YQ", right_on = "datacqtr")
test_comp_macro = test.merge(compustat_macro, how = "left", left_on = "YQ", right_on = "datacqtr")
train_comp_macro.head()

,YQ,NGDP,year,qtr,datacqtr,atq,ltq,niq,atq_lag,ltq_lag,niq_lag,atq_growth,ltq_growth,niq_growth
0,1990Q1,9.0,1990,Q1,1990Q1,3187273.043,224746.034,4684.085,0.000,0.000,0.000,NaN,NaN,NaN
1,1990Q2,6.1,1990,Q2,1990Q2,3412321.689,236377.703,6396.105,3187273.043,224746.034,4684.085,0.070609,0.051755,0.365497
2,1990Q3,3.7,1990,Q3,1990Q3,3473969.743,239062.003,3093.203,3412321.689,236377.703,6396.105,0.018066,0.011356,-0.516393
3,1990Q4,-0.7,1990,Q4,1990Q4,3426461.019,245699.448,-49.699,3473969.743,239062.003,3093.203,-0.013676,0.027765,-1.016067
4,1991Q1,2.0,1991,Q1,1991Q1,3330827.317,296801.210,4647.742,3426461.019,245699.448,-49.699,-0.027910,0.207985,-94.517817


In [40]:
comp_model_atq = smf.ols("NGDP ~ year + qtr + atq", train_comp_macro).fit()
print(comp_model_atq.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.139
Model:                            OLS   Adj. R-squared:                  0.095
Method:                 Least Squares   F-statistic:                     3.154
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0111
Time:                        01:42:50   Log-Likelihood:                -243.71
No. Observations:                 104   AIC:                             499.4
Df Residuals:                      98   BIC:                             515.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -101.0021    222.389     -0.454      0.6

In [42]:
comp_model_atq_lag = smf.ols("NGDP ~ year + qtr + atq_lag", train_comp_macro).fit()
print(comp_model_atq_lag.summary())

y_pred = comp_model_atq_lag.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred, "1_compustat_atqlag_forecast")
# evaluate using RMSE:
print(f"\nRMSE w Assets Lag Macro: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.151
Model:                            OLS   Adj. R-squared:                  0.107
Method:                 Least Squares   F-statistic:                     3.477
Date:                Wed, 04 Mar 2026   Prob (F-statistic):            0.00618
Time:                        01:43:45   Log-Likelihood:                -242.98
No. Observations:                 104   AIC:                             498.0
Df Residuals:                      98   BIC:                             513.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -187.9737    221.947     -0.847      0.3

In [44]:
comp_model_atq_growth = smf.ols("NGDP ~ year + qtr + atq_growth", train_comp_macro).fit()
print(comp_model_atq_growth.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.114
Model:                            OLS   Adj. R-squared:                  0.069
Method:                 Least Squares   F-statistic:                     2.503
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0355
Time:                        01:43:53   Log-Likelihood:                -241.96
No. Observations:                 103   AIC:                             495.9
Df Residuals:                      97   BIC:                             511.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    202.9811     69.390      2.925      0.0

In [45]:
comp_model_ltq = smf.ols("NGDP ~ year + qtr + ltq", train_comp_macro).fit()
print(comp_model_ltq.summary())

y_pred = comp_model_ltq.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred, "2_compustat_ltq_forecast")
# evaluate using RMSE:
print(f"\nRMSE w Liabilities Macro: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.164
Model:                            OLS   Adj. R-squared:                  0.122
Method:                 Least Squares   F-statistic:                     3.849
Date:                Wed, 04 Mar 2026   Prob (F-statistic):            0.00316
Time:                        01:44:48   Log-Likelihood:                -242.14
No. Observations:                 104   AIC:                             496.3
Df Residuals:                      98   BIC:                             512.2
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -76.4702    144.408     -0.530      0.5

In [47]:
comp_model_ltq_lag = smf.ols("NGDP ~ year + qtr + ltq_lag", train_comp_macro).fit()
print(comp_model_ltq_lag.summary())

y_pred = comp_model_ltq_lag.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred, "3_compustat_ltqlag_forecast")
# evaluate using RMSE:
print(f"\nRMSE w Liabilities Lag Macro: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.172
Model:                            OLS   Adj. R-squared:                  0.130
Method:                 Least Squares   F-statistic:                     4.074
Date:                Wed, 04 Mar 2026   Prob (F-statistic):            0.00211
Time:                        01:45:36   Log-Likelihood:                -241.65
No. Observations:                 104   AIC:                             495.3
Df Residuals:                      98   BIC:                             511.2
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -94.6095    141.447     -0.669      0.5

In [48]:
comp_model_ltq_growth = smf.ols("NGDP ~ year + qtr + ltq_growth", train_comp_macro).fit()
print(comp_model_ltq_growth.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.117
Model:                            OLS   Adj. R-squared:                  0.072
Method:                 Least Squares   F-statistic:                     2.577
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0311
Time:                        01:45:44   Log-Likelihood:                -241.79
No. Observations:                 103   AIC:                             495.6
Df Residuals:                      97   BIC:                             511.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    198.5943     69.696      2.849      0.0

In [53]:
comp_model_niq = smf.ols("NGDP ~ year + qtr + niq", train_comp_macro).fit()
print(comp_model_niq.summary())

y_pred = comp_model_niq.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred, "4_compustat_niq_forecast")
# evaluate using RMSE:
print(f"\nRMSE w Net Income Macro: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.345
Model:                            OLS   Adj. R-squared:                  0.312
Method:                 Least Squares   F-statistic:                     10.33
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           5.63e-08
Time:                        01:56:46   Log-Likelihood:                -229.45
No. Observations:                 104   AIC:                             470.9
Df Residuals:                      98   BIC:                             486.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    485.4484     74.782      6.492      0.0

In [54]:
comp_model_niq_lag = smf.ols("NGDP ~ year + qtr + niq_lag", train_comp_macro).fit()
print(comp_model_niq_lag.summary())

y_pred = comp_model_niq_lag.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred, "5_compustat_niqlag_forecast")
# evaluate using RMSE:
print(f"\nRMSE w Net Income Lag Macro: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.301
Model:                            OLS   Adj. R-squared:                  0.265
Method:                 Least Squares   F-statistic:                     8.439
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           1.14e-06
Time:                        01:59:57   Log-Likelihood:                -232.85
No. Observations:                 104   AIC:                             477.7
Df Residuals:                      98   BIC:                             493.6
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    466.2873     78.252      5.959      0.0

In [51]:
comp_model_niq_growth = smf.ols("NGDP ~ year + qtr + niq_growth", train_comp_macro).fit()
print(comp_model_niq_growth.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.120
Model:                            OLS   Adj. R-squared:                  0.074
Method:                 Least Squares   F-statistic:                     2.640
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0278
Time:                        01:46:48   Log-Likelihood:                -241.64
No. Observations:                 103   AIC:                             495.3
Df Residuals:                      97   BIC:                             511.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    212.8997     70.311      3.028      0.0

|Model|RMSE|Adj R-Squared|Score|
|---|---|---|---|
|Base Model RMSE|2.96495609125275|0.083|11.58888|
|RMSE w Assets Lag Macro|3.526561045682736|0.107|11.47981|
|RMSE w Liabilities Macro|3.1749235968328033|0.122|11.63967|
|RMSE w Liabilities Lag Macro|3.122639697340227|0.130|11.52818|
|RMSE w Net Income Macro|3.3948414864271554|0.312|10.76603|
|RMSE w Net Income Lag Macro|3.2575786220077223|0.265|11.71961|

### How Stock Price / Return Data affects GDP (CRSP on WRDS)
1. Essential Variables (The "Core" Signal)
- (dlycaldt) Daily Calendar Date: daily moves into Year-Quarter (YQ) buckets
- (dlytotret) Daily Total Return on Index: This includes dividends. In forensic finance, this is the most powerful leading indicator. It represents the market's "real-time" discount of future economic growth.
- (dlytotval) Index Total Capitalization Value: This is the aggregate dollar value of all firms in the index. Use this to measure the Wealth Effect. If this value rises significantly, consumer spending (the biggest part of GDP) usually follows 1–2 quarters later.

2. Forensic Variables (To Lower Your RMSE)
- (dlytotind) Daily Index Level: Useful if you want to calculate the "momentum" of the market (e.g., comparing the current level to a 200-day moving average).
- (dlyincret) Daily Income Return on Index: This isolates the dividend component. A high income return relative to price return often signals a "flight to quality," which forensically precedes economic slowdowns.
- (dlyusdcnt) Used Count: The number of firms currently in the index. Rapid changes in the count of firms can signal structural shifts in the economy (like a wave of IPOs or bankruptcies).

In [72]:
# Using CRSP Daily Index and Portfolios on S&P 500
# https://wrds-www.wharton.upenn.edu/pages/get-data/center-research-security-prices-crsp/annual-update/index-version-2/daily-index-and-portfolios-on-sp-500/

crsp_daily_macro = pd.read_csv("data/crsp_snp500_indexes.csv")
## dropping columns with no unique data
# print(crsp_daily_macro[crsp_daily_macro["INDNO"] != 1000500].shape)
crsp_daily_macro.drop(columns = "INDNO", inplace = True)

## creating year, month and quarter identifiers
crsp_daily_macro["DlyCalDt"] = pd.to_datetime(crsp_daily_macro["DlyCalDt"])
crsp_daily_macro.set_index("DlyCalDt", inplace = True)
crsp_daily_macro.head()

,DlyTotRet,DlyTotInd,DlyIncRet,DlyUsdCnt,DlyTotVal
DlyCalDt,,,,,
1990-01-02,0.017527,633.75,0.000000,500,2371902979
1990-01-03,-0.002028,632.47,0.000020,500,2367046297
1990-01-04,-0.007913,627.46,0.000434,500,2347288970
1990-01-05,-0.010069,621.14,0.000023,500,2323602293
1990-01-08,0.004798,624.12,0.000088,500,2334547314


Since we need to convert the data from daily values to quarterly values, we can use `resample()`<br>
Create a `YQ` column for easier merge, then we can drop `DlyCalDt`<br>
After creating the aggregate quarterly values, create the lag as market signals to predict current quarter

In [73]:
## create quarterly data
crsp_qtr_macro = crsp_daily_macro.resample("Q").agg({
    "DlyTotRet": [
        ("qtr_ret", lambda x: (1+x).prod() - 1),    # compounded total return
        ("qtr_vola", "std")                         # quarterly volatility
    ],
    "DlyTotInd": [("qtr_idx_level", "last")],       # final index level
    "DlyIncRet": [("qtr_income_ret", "sum")],       # total income return on index
    "DlyUsdCnt": [("avg_firm_count", "mean")]       # average number of firms
})

crsp_qtr_macro.columns = crsp_qtr_macro.columns.droplevel(0)
crsp_qtr_macro.reset_index(inplace = True)
## creating YQ column and moving it to first column
crsp_qtr_macro["YQ"] = crsp_qtr_macro["DlyCalDt"].dt.year.astype(str) + "Q" + crsp_qtr_macro["DlyCalDt"].dt.quarter.astype(str)
crsp_qtr_macro.drop(columns = "DlyCalDt", inplace = True)
col = crsp_qtr_macro.pop("YQ")
crsp_qtr_macro.insert(0, "YQ", col)
crsp_qtr_macro.sort_values(by = "YQ", inplace = True)
market_cols = "qtr_ret", "qtr_vola", "qtr_idx_level", "qtr_income_ret", "avg_firm_count"
# creating last column market signals to predict current quarter
for col in market_cols: crsp_qtr_macro[f"{col}_lag"] = crsp_qtr_macro[col].shift(1)
crsp_qtr_macro

C:\Users\sbgka\AppData\Local\Temp\ipykernel_26536\3879900254.py:2: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  crsp_qtr_macro = crsp_daily_macro.resample("Q").agg({


,YQ,qtr_ret,qtr_vola,qtr_idx_level,qtr_income_ret,avg_firm_count,qtr_ret_lag,qtr_vola_lag,qtr_idx_level_lag,qtr_income_ret_lag,avg_firm_count_lag
0,1990Q1,-0.030222,0.008859,604.01,0.008371,499.984127,NaN,NaN,NaN,NaN,NaN
1,1990Q2,0.063453,0.007631,642.34,0.008977,500.000000,-0.030222,0.008859,604.01,0.008371,499.984127
2,1990Q3,-0.139556,0.011900,552.70,0.008717,499.984127,0.063453,0.007631,642.34,0.008977,500.000000
3,1990Q4,0.090365,0.011030,602.64,0.010018,500.000000,-0.139556,0.011900,552.70,0.008717,499.984127
4,1991Q1,0.147108,0.010541,691.30,0.008023,500.000000,0.090365,0.011030,602.64,0.010018,500.000000
...,...,...,...,...,...,...,...,...,...,...,...
119,2019Q4,0.089929,0.005923,10872.55,0.005012,505.000000,0.016904,0.009357,9975.45,0.005074,504.890625
120,2020Q1,-0.194095,0.035693,8762.26,0.005267,504.967742,0.089929,0.005923,10872.55,0.005012,505.000000
121,2020Q2,0.206679,0.019921,10573.22,0.004871,505.000000,-0.194095,0.035693,8762.26,0.005267,504.967742
122,2020Q3,0.089977,0.010664,11524.56,0.004149,505.000000,0.206679,0.019921,10573.22,0.004871,505.000000


In [74]:
# merging updated macro to train and test sets
train_crsp_macro = train.merge(crsp_qtr_macro, on = "YQ", how = "left")
test_crsp_macro = test.merge(crsp_qtr_macro, on = "YQ", how = "left")
train_crsp_macro.head()

,YQ,NGDP,year,qtr,qtr_ret,qtr_vola,qtr_idx_level,qtr_income_ret,avg_firm_count,qtr_ret_lag,qtr_vola_lag,qtr_idx_level_lag,qtr_income_ret_lag,avg_firm_count_lag
0,1990Q1,9.0,1990,Q1,-0.030222,0.008859,604.01,0.008371,499.984127,NaN,NaN,NaN,NaN,NaN
1,1990Q2,6.1,1990,Q2,0.063453,0.007631,642.34,0.008977,500.000000,-0.030222,0.008859,604.01,0.008371,499.984127
2,1990Q3,3.7,1990,Q3,-0.139556,0.011900,552.70,0.008717,499.984127,0.063453,0.007631,642.34,0.008977,500.000000
3,1990Q4,-0.7,1990,Q4,0.090365,0.011030,602.64,0.010018,500.000000,-0.139556,0.011900,552.70,0.008717,499.984127
4,1991Q1,2.0,1991,Q1,0.147108,0.010541,691.30,0.008023,500.000000,0.090365,0.011030,602.64,0.010018,500.000000


## STOPPED HERE

In [75]:
crsp_model_ret = smf.ols("NGDP ~ year + qtr + qtr_ret", train_crsp_macro).fit()
print(crsp_model_ret.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.194
Model:                            OLS   Adj. R-squared:                  0.153
Method:                 Least Squares   F-statistic:                     4.727
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           0.000654
Time:                        02:46:29   Log-Likelihood:                -240.23
No. Observations:                 104   AIC:                             492.5
Df Residuals:                      98   BIC:                             508.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    207.6220     65.868      3.152      0.0

In [76]:
crsp_model_ret_lag = smf.ols("NGDP ~ year + qtr + qtr_ret_lag", train_crsp_macro).fit()
print(crsp_model_ret_lag.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.218
Model:                            OLS   Adj. R-squared:                  0.177
Method:                 Least Squares   F-statistic:                     5.400
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           0.000200
Time:                        02:46:30   Log-Likelihood:                -235.57
No. Observations:                 103   AIC:                             483.1
Df Residuals:                      97   BIC:                             498.9
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept     187.4154     65.351      2.868      

In [77]:
crsp_model_vola = smf.ols("NGDP ~ year + qtr + qtr_vola", train_crsp_macro).fit()
print(crsp_model_vola.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.391
Model:                            OLS   Adj. R-squared:                  0.360
Method:                 Least Squares   F-statistic:                     12.60
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           1.87e-09
Time:                        02:46:30   Log-Likelihood:                -225.65
No. Observations:                 104   AIC:                             463.3
Df Residuals:                      98   BIC:                             479.2
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    157.8598     57.905      2.726      0.0

In [78]:
crsp_model_vola_lag = smf.ols("NGDP ~ year + qtr + qtr_vola_lag", train_crsp_macro).fit()
print(crsp_model_vola_lag.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.294
Model:                            OLS   Adj. R-squared:                  0.258
Method:                 Least Squares   F-statistic:                     8.085
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           2.08e-06
Time:                        02:46:30   Log-Likelihood:                -230.27
No. Observations:                 103   AIC:                             472.5
Df Residuals:                      97   BIC:                             488.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      150.7093     62.827      2.399   

In [79]:
crsp_model_index = smf.ols("NGDP ~ year + qtr + qtr_idx_level", train_crsp_macro).fit()
print(crsp_model_index.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.200
Model:                            OLS   Adj. R-squared:                  0.160
Method:                 Least Squares   F-statistic:                     4.909
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           0.000473
Time:                        02:46:30   Log-Likelihood:                -239.84
No. Observations:                 104   AIC:                             491.7
Df Residuals:                      98   BIC:                             507.6
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept       663.8994    154.725      4.291

In [80]:
crsp_model_index_lag = smf.ols("NGDP ~ year + qtr + qtr_idx_level_lag", train_crsp_macro).fit()
print(crsp_model_index_lag.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.141
Model:                            OLS   Adj. R-squared:                  0.097
Method:                 Least Squares   F-statistic:                     3.192
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0104
Time:                        02:46:30   Log-Likelihood:                -240.37
No. Observations:                 103   AIC:                             492.7
Df Residuals:                      97   BIC:                             508.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           456.3901    160.17

In [81]:
crsp_model_inc_ret = smf.ols("NGDP ~ year + qtr + qtr_income_ret", train_crsp_macro).fit()
print(crsp_model_inc_ret.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.201
Model:                            OLS   Adj. R-squared:                  0.160
Method:                 Least Squares   F-statistic:                     4.930
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           0.000456
Time:                        02:46:30   Log-Likelihood:                -239.80
No. Observations:                 104   AIC:                             491.6
Df Residuals:                      98   BIC:                             507.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        300.7750     70.240      4.

In [82]:
crsp_model_inc_ret_lag = smf.ols("NGDP ~ year + qtr + qtr_income_ret_lag", train_crsp_macro).fit()
print(crsp_model_inc_ret_lag.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.164
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     3.793
Date:                Wed, 04 Mar 2026   Prob (F-statistic):            0.00351
Time:                        02:46:30   Log-Likelihood:                -239.02
No. Observations:                 103   AIC:                             490.0
Df Residuals:                      97   BIC:                             505.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            266.8657     72

Insights: average firm count is statistically insignificant to the Nominal GDP

In [83]:
crsp_model_firm_cnt = smf.ols("NGDP ~ year + qtr + avg_firm_count", train_crsp_macro).fit()
print(crsp_model_firm_cnt.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.119
Model:                            OLS   Adj. R-squared:                  0.074
Method:                 Least Squares   F-statistic:                     2.645
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0275
Time:                        02:46:30   Log-Likelihood:                -244.89
No. Observations:                 104   AIC:                             501.8
Df Residuals:                      98   BIC:                             517.6
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        264.3963    205.705      1.

In [84]:
crsp_model_firm_cnt_lag = smf.ols("NGDP ~ year + qtr + avg_firm_count_lag", train_crsp_macro).fit()
print(crsp_model_firm_cnt_lag.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.114
Model:                            OLS   Adj. R-squared:                  0.069
Method:                 Least Squares   F-statistic:                     2.505
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0353
Time:                        02:46:30   Log-Likelihood:                -241.96
No. Observations:                 103   AIC:                             495.9
Df Residuals:                      97   BIC:                             511.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            238.4854    301